**Vector to raster**

Convert a shapefile to a raster

In [30]:
# Load libraries

import os
import geopandas as gpd
import rasterio
import numpy as np
import matplotlib.pyplot as plt
from rasterio.features import rasterize
from google.colab import drive
from rasterio.transform import from_origin
from shapely.validation import make_valid
from shapely.geometry import Point
from rasterio.transform import from_bounds
from matplotlib.colors import ListedColormap, BoundaryNorm

In [22]:
# Mount the drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [23]:
# Specify link to directory where raster files are saved

peatmap_shapefile = '/content/drive/MyDrive/GitHub_Desktop/Work/Data/PEATMAP_Global'

In [24]:
# List the files in the directory

file_paths = os.listdir(peatmap_shapefile)
file_paths

['PEATMAP_Global.cpg',
 'PEATMAP_Global.prj',
 'PEATMAP_Global.qmd',
 'PEATMAP_Global.dbf',
 'PEATMAP_Global.shp',
 'PEATMAP_Global.shx']

In [25]:
# Load shapefile & set output raster

vector_path = '/content/drive/MyDrive/GitHub_Desktop/Work/Data/PEATMAP_Global/PEATMAP_Global.shp'
output_raster = '/content/drive/MyDrive/GitHub_Desktop/Work/Data/PEATMAP_Global/PEATMAP_Global.tif'

# Read shapefile
gdf = gpd.read_file(vector_path)

# Print metadata
print("CRS:", gdf.crs)
print("Fields:", gdf.columns)
print("Number of features:", len(gdf))

CRS: EPSG:4326
Fields: Index(['PEAT_AREA', 'Area', 'PEAT_PER', 'Id', 'layer', 'path', 'geometry'], dtype='object')
Number of features: 48697


In [26]:
print(gdf[attribute_field].unique())
print(gdf[attribute_field].describe())

['AF_Peatland' 'EA_Peatland' 'Histosols_Hokkaido_Mongolia_North Korea'
 'NEA_Peatland' 'Oceania_Peatland' 'SA_Peatland' 'SEA_Peatland'
 'SIB_Peatland']
count           48697
unique              8
top       AF_Peatland
freq            20412
Name: layer, dtype: object


In [33]:
# Check if CRS is geographic
print(f"Current CRS: {gdf.crs}")
print(f"Is geographic: {gdf.crs.is_geographic if gdf.crs else 'No CRS set'}")

# If geographic, reproject to projected CRS
if gdf.crs and gdf.crs.is_geographic:
    print("Converting from geographic to projected CRS...")

    # Option 1: Web Mercator (global, but distorted at high latitudes)
    gdf = gdf.to_crs('EPSG:3857')

    # Option 2: Estimate best UTM zone (better accuracy)
    # from pyproj import CRS
    # centroid = gdf.dissolve().centroid.iloc[0]
    # utm_zone = int((centroid.x + 180) / 6) + 1
    # hemisphere = 'north' if centroid.y >= 0 else 'south'
    # utm_crs = CRS.from_dict({'proj': 'utm', 'zone': utm_zone, 'south': hemisphere == 'south'})
    # gdf = gdf.to_crs(utm_crs)

    print(f"New CRS: {gdf.crs}")

# Now recalculate with projected coordinates
minx, miny, maxx, maxy = gdf.total_bounds
pixel_size = 30  # Now in meters
width = int(np.ceil((maxx - minx) / pixel_size))
height = int(np.ceil((maxy - miny) / pixel_size))
print(f"Dimensions: {width} x {height}")

Current CRS: EPSG:4326
Is geographic: True
Converting from geographic to projected CRS...
New CRS: EPSG:3857
Dimensions: 1335834 x 667626


In [ ]:
# Rasterize

# Get bounds from your GeoDataFrame
minx, miny, maxx, maxy = gdf.total_bounds

# Define pixel size (in the same units as your CRS)
pixel_size = 1000  # meters if using a projected CRS

# Calculate raster dimensions
width = int((maxx - minx) / pixel_size)
height = int((maxy - miny) / pixel_size)

# Create the affine transform
transform = from_bounds(minx, miny, maxx, maxy, width, height)

# Prepare shapes for rasterization
attribute_field = "layer"

shapes = (
    (geom, value)
    for geom, value in zip(gdf.geometry, gdf[attribute_field])
)

# Rasterize
raster = rasterize(
    shapes=shapes,
    out_shape=(height, width),
    transform=transform,
    fill=0,
    dtype="uint8"
)

print(f"Raster shape: {raster.shape}")
print(f"Raster bounds: ({minx}, {miny}, {maxx}, {maxy})")
print(f"Pixel size: {pixel_size}")

In [ ]:
# Save to GeoTIFF

with rasterio.open(
    output_raster,
    'w',
    driver='GTiff',
    height=height,
    width=width,
    count=1,
    dtype=raster.dtype,
    crs=gdf.crs,
    transform=transform,
    compress='lzw'
) as dst:
    dst.write(raster, 1)

print("Rasterization complete!")